In [1]:
"""
Dynamic Airline Factor Laboratory
_______
Orchestrates the full research pipeline from raw data to output CSVs.

Pipeline
_______
    [1]  Download and clean market data
    [2]  Split into training (2015–2019) and validation (2020–2024) periods
    [3]  Window sensitivity analysis on training period (60, 90, 120, 180 days)
    [4]  Select optimal window; hold fixed for validation
    [5]  Run full factor model on selected window
    [6]  Report variance explained by PC1–PCk (diagnose n_components)
    [7]  Generate signals → monthly rebalance → construct portfolio
    [8]  Compute PnL with 15 bps transaction costs
    [9]  Evaluate performance: Sharpe, drawdown, alpha/beta vs JETS and SPY
    [10] Macro regime diagnostics (VIX, Brent quartiles)
    [11] Write all output CSVs to outputs/

Train / Test Discipline
_______
    Training  : 2015-01-01 → 2019-12-31
        - Window length selected here (60, 90, 120, 180 days)
        - n_components tested here (1, 2, 3)
        - No other parameters tuned

    Validation: 2022-01-01 → 2026-07-17
        - Selected window applied unchanged
        - All performance statistics reported from this period
"""


'\nDynamic Airline Factor Laboratory\n_______\nOrchestrates the full research pipeline from raw data to output CSVs.\n\nPipeline\n_______\n    [1]  Download and clean market data\n    [2]  Split into training (2015–2019) and validation (2020–2024) periods\n    [3]  Window sensitivity analysis on training period (60, 90, 120, 180 days)\n    [4]  Select optimal window; hold fixed for validation\n    [5]  Run full factor model on selected window\n    [6]  Report variance explained by PC1–PCk (diagnose n_components)\n    [7]  Generate signals → monthly rebalance → construct portfolio\n    [8]  Compute PnL with 15 bps transaction costs\n    [9]  Evaluate performance: Sharpe, drawdown, alpha/beta vs JETS and SPY\n    [10] Macro regime diagnostics (VIX, Brent quartiles)\n    [11] Write all output CSVs to outputs/\n\nTrain / Test Discipline\n_______\n    Training  : 2015-01-01 → 2019-12-31\n        - Window length selected here (60, 90, 120, 180 days)\n        - n_components tested here (1, 2,

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import data as dt
import models as md
import backtest as bt
output_dir = Path("../Outputs")
output_dir.mkdir(exist_ok=True)

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
prices_norm = prices / prices.iloc[0]
plt.figure(figsize=(16,8))
for col in prices_norm.columns:plt.plot(prices_norm.index,prices_norm[col],label=col,linewidth=1)
plt.legend(ncol=2)
plt.title("Normalized Price Series")
plt.grid(True)
plt.show()

In [ ]:
print("Parameters")
print("_______")
print("Training End :", dt.train_end)
print("Validation Start :", dt.test_start)
print("Lookback :", md.lookback)
print("Z Window :", md.zscore_window)
print("Entry :", bt.entry_threshold)
print("Exit :", bt.exit_threshold)
print("PCs :", md.n_components)

In [ ]:
airline_returns, macro_returns, prices = dt.load_data()
display(prices.head())
display(airline_returns.head())
display(macro_returns.head())

In [ ]:
train_mask = airline_returns.index <= dt.train_end
test_mask = airline_returns.index >= dt.test_start
train_airlines = airline_returns.loc[train_mask]
test_airlines = airline_returns.loc[test_mask]
train_macro = macro_returns.loc[train_mask]
test_macro = macro_returns.loc[test_mask]
print(train_airlines.shape)
print(test_airlines.shape)

In [ ]:
window_results = []
for w in [30,45,60,75,90,120]:
    cov = md.rolling_covariances(train_airlines, window=w)
    _, residuals, _, _, _ = md.compute_factor_model(train_airlines, cov, md.n_components)
    z = md.compute_rolling_zscore(residuals).dropna()
    sig = bt.generate_signals(z)
    pos = bt.construct_portfolio(sig)
    net, trades = bt.compute_portfolio_returns(pos, train_airlines)
    perf = bt.compute_performance(net, trades, train_macro)
    window_results.append([w, perf["sharpe_ratio"]])
window_results = pd.DataFrame(window_results, columns=["Window","Sharpe"])
display(window_results)

In [ ]:
best_window = 60

In [ ]:
cov = md.rolling_covariances(train_airlines, window=best_window)

In [ ]:
zscores = md.compute_rolling_zscore(residuals)

In [ ]:
signals = bt.generate_signals(zscores)
positions = bt.construct_portfolio(signals)

In [ ]:
positions.tail()

In [ ]:
net,trades = bt.compute_portfolio_returns(positions,train_airlines)

In [ ]:
perf = bt.compute_performance(net,trades,train_macro)
display(perf)